In [1]:
import pandas as pd
from pathlib import Path
import sys
from IPython.display import display, Markdown

In [2]:
sys.path.append("../src")
from explainer import ForensicExplainer

In [7]:
BASE_DIR = Path.cwd().parent
CHATS_CSV = BASE_DIR / "data" / "processed" / "chats_completos.csv"
NODES_FEATURES_CSV = BASE_DIR / "data" / "processed" / "nodos_features.csv"
ANOMALIES_CSV = BASE_DIR / "data" / "processed" / "chats_top.csv"

In [8]:
explainer = ForensicExplainer(model_name="llama-3.3-70b-versatile")

df_chats = pd.read_csv(CHATS_CSV)
df_features = pd.read_csv(NODES_FEATURES_CSV)
df_anomalies = pd.read_csv(ANOMALIES_CSV)

In [5]:
# SECCIÓN 1: EXPLICABILIDAD DE GRAFOS
display(Markdown("## 🧠 1. Análisis Estructural y Perfilado de Actores Clave"))
display(Markdown("Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red."))

# Filtramos los 5 actores principales
top_actores = df_features.nlargest(5, 'PageRank')

for _, row in top_actores.iterrows():
    actor = row['Nodo']
    rol_asignado = row.get('Etiqueta_Forense', 'Rol Desconocido')
    
    # Extraemos hasta 5 mensajes largos del actor para dar contexto
    mensajes_actor = df_chats[df_chats['Emisor'].str.upper().str.contains(str(actor).upper(), na=False)]['Mensaje'].dropna().tolist()
    mensajes_muestra = sorted(mensajes_actor, key=len, reverse=True)[:5]
    
    if mensajes_muestra:
        explicacion = explainer.explain_actor_profile(actor, rol_asignado, mensajes_muestra)
        
        display(Markdown(f"### 👤 Actor: `{actor}`"))
        display(Markdown(f"- **Clasificación Matemática:** `{rol_asignado}`"))
        display(Markdown(f"- **Evaluación Forense (LLM):**\n> {explicacion}\n"))
        display(Markdown("---"))

## 🧠 1. Análisis Estructural y Perfilado de Actores Clave

Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red.

### 👤 Actor: `RODOLFO REYES`

- **Clasificación Matemática:** `Rol Desconocido`

- **Evaluación Forense (LLM):**
> La información proporcionada sugiere que Rodolfo REYES participa en una red de influencias con un rol desconocido, pero su nivel de involucramiento y acceso a información sensible indica una posición de cierta jerarquía. Maneja información relacionada con ayudas públicas y préstamos, y mantiene comunicaciones con individuos clave en la red. Su interacción con Julio MARTINEZ SOLA y Roberto ROSELLI revela un nivel de confianza y acceso a información privilegiada. Esto sugiere que su rol en la red puede ser más significativo de lo que inicialmente se desconoce.


...

---

### 👤 Actor: `JULIO MARTINEZ MARTINEZ`

- **Clasificación Matemática:** `Rol Desconocido`

- **Evaluación Forense (LLM):**
> El contenido de los mensajes de "JULIO MARTINEZ MARTINEZ" sugiere un nivel de jerarquía elevado, con menciones a reuniones y aprobaciones. La información que maneja parece estar relacionada con temas económicos y empresariales, incluyendo rescates y participaciones estatales. El tono y el lenguaje utilizado indican un ambiente formal y profesional. El rol de "Rol Desconocido" podría corresponder a un cargo de influencia en el ámbito económico o empresarial.


---

In [9]:
# SECCIÓN 2: EXPLICABILIDAD DE ANOMALÍAS TEMPORALES
display(Markdown("---"))
display(Markdown("## 🚨 2. Análisis Semántico de Anomalías de Comportamiento"))
display(Markdown("Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest."))

# Ordenamos por Anomaly_Score (los más negativos son los más anómalos)
top_anomalias = df_anomalies.sort_values(by='Anomaly_Score', ascending=True).head(5)

for i, row in top_anomalias.iterrows():
    fecha = str(row['Datetime'])
    hora = f"{row['Hour']}:00"
    texto = str(row['Mensaje'])
    longitud = row['Message_Length']
    score = row['Anomaly_Score']
    
    explicacion_anomalia = explainer.explain_anomaly(fecha, hora, longitud, score, texto)
    
    display(Markdown(f"### ⚠️ Anomalía Crítica Nivel {i+1}"))
    display(Markdown(f"- **Metadatos:** `Fecha: {fecha} | Score: {score:.3f} | Longitud: {longitud} chars`"))
    display(Markdown(f"- **Texto Base:** _{texto[:150]}..._"))
    display(Markdown(f"- **Evaluación Forense (LLM):**\n> {explicacion_anomalia}\n"))
    display(Markdown("---"))

---

## 🚨 2. Análisis Semántico de Anomalías de Comportamiento

Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest.

### ⚠️ Anomalía Crítica Nivel 1

- **Metadatos:** `Fecha: 2020-11-04 09:15:19 | Score: -0.056 | Longitud: 1149 chars`

...

### ⚠️ Anomalía Crítica Nivel 5

- **Metadatos:** `Fecha: 2016-07-21 01:53:51 | Score: -0.034 | Longitud: 23 chars`

- **Texto Base:** _Moncho Gordils por aqui..._

- **Evaluación Forense (LLM):**
> El evento presenta una anomalía matemática extrema con un score de -0.034, lo que sugiere un patrón inusual. La longitud inusual de 23 caracteres y el contenido del mensaje "Moncho Gordils por aqui" pueden indicar una coordinación clandestina o un mensaje codificado. El contexto y la hora del evento (01:53:51) podrían sugerir una comunicación secreta. La evaluación forense sugiere que este evento requiere una investigación más profunda para determinar su naturaleza y posible impacto.


---